In [ ]:
import time
import os

import pandas as pd
import numpy as np
from datetime import datetime

pd.set_option("display.max_columns", None)

ARQUIVO_AUX = "abfss://96fe5a53-3a22-4443-8d0a-e2f6d61a2690@onelake.dfs.fabric.microsoft.com/0f8d9b0e-86cc-4454-9772-4ab92eb4db2a/Files/acto/tb_aux.xlsx"


def ajustar_nome_colunas(df):
    # Colunas em letras minúsculas e sem acentos/símbolos
    df.columns = (
        df.columns.str.lower()
        .str.strip()
        .str.replace("  ", " ")
        .str.replace("º", "")
        .str.replace(":", "")
        .str.replace("ç", "c")
        .str.replace("ã", "a")
        .str.replace("ú", "u")
        .str.replace("ê", "e")
        .str.replace("á", "a")
        .str.replace(r"\s+", "_", regex=True)
    )

    # df = df.loc[:, ~df.columns.duplicated()]
    return df



def processar_os():

    os_santos = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ";")
        .load("Files/acto/exportar.csv")
    ).toPandas()

    # Remove duplicatas pela coluna de número da solicitação
    os_santos = os_santos.drop_duplicates(subset="Nº da Solicitação", keep="first")

    # Unifica colunas de Canal:
    os_santos["canal"] = np.where(
        os_santos["Canal"].isna(), os_santos["Canal:"], os_santos["Canal"]
    )
    # Preenche canal com 'Presencial' se vazio
    os_santos["canal"] = os_santos["canal"].replace("", np.nan).fillna("Presencial")

    # Remove e renomeia colunas Canal
    os_santos = os_santos.drop(columns=[
        "Canal", "Canal:", "Nome do Serviço", 
        "Serviço:", "Etapa Atual:", "Executor Atual:", 
        "Data de Solicitação:", "Data de Finalização:",
        "Status:", "Detalhes da solicitação",
        "Nº da Solicitação:", "Nome do serviço:"
    ])
    os_santos = os_santos.rename(columns={"Bairro:":"bairro_cet"})
    os_santos = ajustar_nome_colunas(os_santos)  # Padroniza colunas

    os_santos = os_santos.rename(columns={"nome_do_servico_-_manifestacao": "nome_do_servico"})

    cols_bairro_interessado = ['bairro_interessado50','bairro_interessado62']
    os_santos['bairro_interessado'] = os_santos[cols_bairro_interessado].bfill(axis=1).iloc[:, 0]

    cols_bairro_ocorrencia  = [
        'bairro_ocorrencia65', 'bairro_ocorrencia71'
    ]
    os_santos['bairro_ocorrencia'] = os_santos[cols_bairro_ocorrencia].bfill(axis=1).iloc[:, 0]

    # Capitaliza algumas colunas
    for col in [
        "solicitante",
        "etapa_atual",
        "servico",
    ]:
        os_santos[col] = os_santos[col].str.capitalize()

    for col in [
        "solicitante",
        "nome_interessado",
        "executor_atual",
        "bairro_interessado",
        "bairro_ocorrencia",
        "bairro",
        "bairro_cet"
    ]:
        os_santos[col] = os_santos[col].str.title()

    # Padroniza serviços específicos
    for col in ["servico", "nome_do_servico"]:
        os_santos[col] = (
            os_santos[col]
            .str.capitalize()
            .str.replace(
                "Serviço - poda de raiz de árvore",
                "Poda de raiz de árvore",
            )
            .str.replace("Tapa buraco", "Tapa-buraco")
        )
    # Cria coluna consolidada de bairro
    os_santos["bairro_consolidado"] = np.where(
        os_santos["bairro"].notna(),
        os_santos["bairro"],
        np.where(
            os_santos["bairro_cet"].notna(),
            os_santos["bairro_cet"],
            np.where(
                os_santos["bairro_ocorrencia"].notna(),
                os_santos["bairro_ocorrencia"],
                os_santos["bairro_interessado"],
            )
        )
    )

    # Criar coluna consolidada do nome do interessado
    os_santos["nome_interessado"] = np.where(
        os_santos["nome_do_interessado20"].notna(),
        os_santos["nome_do_interessado20"],
        os_santos["nome_interessado"],
    )

    os_santos = os_santos.drop(columns="vencimento_sla")

    os_santos['vencimento_sla'] = np.nan

    return os_santos


def processar_prazo():
    prazo = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_prazo")
    prazo = ajustar_nome_colunas(prazo)
    prazo["servico"] = prazo["servico"].str.capitalize()
    return prazo


def processar_bairros():
    bairros = pd.read_excel(ARQUIVO_AUX, sheet_name="aux_regionais")
    bairros = ajustar_nome_colunas(bairros)
    bairros["bairro_consolidado"] = bairros["bairro"].str.title()
    bairros = bairros.drop(columns=["bairro"])
    return bairros


def harmonizar_ordenar_etapas(df):

    df["etapa_atual"] = (
        df["etapa_atual"]
        .str.strip()
        .str.title()
        .str.replace("Análise De Área", "Análise Da Área", regex=False)
        .str.replace("Vistoria/Análise", "Vistoria E Análise", regex=False)
        .str.replace("Vistoria / Análise", "Vistoria E Análise", regex=False)
        .str.replace("Análise Execução", "Análise De Execução", regex=False)
        .str.replace("Avaliação (Ouvidoria)", "Avaliação", regex=False)
    )

    ordem_etapas = {
        "Abertura": 1,
        "Triagem": 2,
        "Atendente": 3,
        "Análise Da Área": 4,
        "Comunicação": 5,
        "Vistoria E Análise": 6,
        "Aguardando Programação": 7,
        "Aguardando Execução Programada": 8,
        "Execução Sepref": 9,
        "Execução Prodesan": 10,
        "Execução Terceiro": 11,
        "Análise De Execução": 12,
        "Manifestação": 13,
        "Classificação": 14,
        "Complementação De Dados": 15,
        "Resposta Da Secretaria": 16,
        "Resposta Ao Cidadão": 17,
        "Mediação": 18,
        "Avaliação": 19,
    }

    df["ordem_etapa"] = df["etapa_atual"].map(ordem_etapas).fillna(99).astype(int)

    return df


def aplicar_merge(acto, prazo, bairros):
    # Mescla dados (JOIN)
    acto_prazo = pd.merge(acto, prazo, on="servico", how="inner")
    acto_prazo = pd.merge(acto_prazo, bairros, on="bairro_consolidado", how="left")
    return acto_prazo


def tratar_datas(acto_prazo):

    # Trata datas
    acto_prazo["data_de_solicitacao"] = pd.to_datetime(
        acto_prazo["data_de_solicitacao"], dayfirst=True
    )
    acto_prazo["data_de_finalizacao"] = pd.to_datetime(
        acto_prazo["data_de_finalizacao"], dayfirst=True
    )

    # Calcula vencimento do prazo
    acto_prazo["data_de_vencimento_prazo"] = acto_prazo[
        "data_de_solicitacao"
    ] + pd.to_timedelta(acto_prazo["prazo_de_conclusao"], unit="D")

    # Calcula tempo de execução se finalizado
    acto_prazo["tempo_de_execucao_real"] = np.where(
        acto_prazo["status"] == "Finalizado",
        (acto_prazo["data_de_finalizacao"] - acto_prazo["data_de_solicitacao"]).dt.days,
        0,
    )
    acto_prazo["tempo_de_execucao_real"] = (
        acto_prazo["tempo_de_execucao_real"].astype(int)
    )

    # Data de hoje como datetime apenas com data (sem hora)
    data_hoje = pd.to_datetime(datetime.now().date())

    # Calcula dias até vencimento
    acto_prazo["dias_ate_vencimento"] = (
        acto_prazo["data_de_vencimento_prazo"] - data_hoje
    ).dt.days

    acto_prazo["dias_ate_vencimento"] = np.where(
        acto_prazo["status"].isin(["Finalizado", "Cancelado"]),
        0,
        acto_prazo["dias_ate_vencimento"]
    )

    acto_prazo['mes_solicitacao'] = acto_prazo['data_de_solicitacao'].dt.month
    acto_prazo['ano_solicitacao'] = acto_prazo['data_de_solicitacao'].dt.year

    acto_prazo['mes_finalizacao'] = acto_prazo['data_de_finalizacao'].dt.month
    acto_prazo['ano_finalizacao'] = acto_prazo['data_de_finalizacao'].dt.year

    acto_prazo['mes_vencimento_prazo'] = acto_prazo['data_de_vencimento_prazo'].dt.month
    acto_prazo['ano_vencimento_prazo'] = acto_prazo['data_de_vencimento_prazo'].dt.year

    # Define status do prazo de execução com 3 opções ajustadas para não exibir "Vence hoje" em OS finalizadas
    # Normalizar as datas para comparar apenas a parte da data (sem horário)
    data_hoje_normalizada = pd.to_datetime(datetime.now().date())
    data_vencimento_normalizada = pd.to_datetime(acto_prazo["data_de_vencimento_prazo"]).dt.date
    data_finalizacao_normalizada = pd.to_datetime(acto_prazo["data_de_finalizacao"]).dt.date

    acto_prazo["status_conclusao_servico"] = np.where(
        # Se o status da OS for "Cancelado", marca como "Cancelado"
        acto_prazo["status"].str.lower() == "cancelado",
        "Cancelado",
        np.where(
            # Se o serviço foi finalizado (tem data de finalização)
            acto_prazo["data_de_finalizacao"].notna(),
            # Verificar se foi entregue dentro ou fora do prazo
            np.where(
                data_finalizacao_normalizada <= data_vencimento_normalizada,
                "Dentro do prazo",  # Entregue dentro do prazo
                "Fora do prazo"     # Entregue fora do prazo
            ),
            # Se não foi finalizado (data_finalizacao é nula) - ainda em andamento
            np.where(
                data_vencimento_normalizada > data_hoje_normalizada.date(),
                "Dentro do prazo",  # Ainda em andamento e dentro do prazo
                np.where(
                    data_vencimento_normalizada == data_hoje_normalizada.date(),
                    "Vence hoje",    # Ainda em andamento e vence hoje
                    "Vencido"        # Ainda em andamento mas vencido
                )
            )
        )
    )


    for date_col in ['data_de_solicitacao', 'data_de_finalizacao', 'data_de_vencimento_prazo']:
        acto_prazo[date_col] = acto_prazo[date_col].dt.date

    return acto_prazo


def remover_registros_teste(acto_prazo):
    # Remove registros de solicitantes testes ou indevidos
    lista = [
        "André Ygor Bulata Dos Santos",
        # "Administraacto Administraacto",
        "Matheus De Paula Moura Arsenes",
        "Wagner De Morais Pechim",
        "Teste M",
        "Teste Matheus",
        "Teste Teste",
        "Dialla Araujo Souza",
        "Victor Martins Da Silva",
        "Yago Silva De Jesus",
        "Matheus Santos Alves",
        "Milena Firmiano Lopes",
        "Arleque Sandra Aparecida De Souza",
    ]

    acto_prazo["solicitante"] = acto_prazo["solicitante"].str.strip().str.title()
    acto_prazo = acto_prazo.loc[~acto_prazo["solicitante"].isin(lista)]

    return acto_prazo


def tratar_base_final_solicitacoes(acto_prazo):

    # Atribui unidade executora (COPAISA ou região)
    servicos_copaisa = [
        "Corte de grama",
        "Avaliação técnica de árvores",
        "Poda de copa de árvore",
        "Poda de raiz de árvore",
        "Remoção de árvores",
    ]

    acto_prazo["unidade_executora"] = np.where(
        (acto_prazo["nome_do_servico"].isin(servicos_copaisa))
        | (acto_prazo["servico"].isin(servicos_copaisa)),
        "COPAISA",
        np.where(
            acto_prazo["secretaria"].str.contains("CET"),
            acto_prazo["secretaria"],
            acto_prazo["regiao"],
        ),
    )
    acto_prazo["unidade_executora"] = np.where(
        acto_prazo["secretaria"].str.contains('SEGOV'),
        "SEALURB",
        acto_prazo["unidade_executora"]
    )

    # Caso ainda reste algum valor em branco, atribui rótulo padrão
    acto_prazo["unidade_executora"] = acto_prazo["unidade_executora"].fillna(
        "Não informado"
    )


    # Define responsavel pela execução dos serviços da SEINFRA
    acto_prazo["responsavel_execucao"] = np.where(
        acto_prazo['etapa_atual'] == 'Execução Terceiro',
        "Empresa terceira",
        acto_prazo["responsavel_execucao"]
    )
    acto_prazo["responsavel_execucao"] = np.where(
        acto_prazo["secretaria"].str.contains("SEINFRA")
        & acto_prazo["status"].isin(["Em atendimento", "Pendente"])
        & acto_prazo["responsavel_execucao"].isna(),
        "A ser definido",
        acto_prazo["responsavel_execucao"]
    )
    acto_prazo["responsavel_execucao"] = np.where(
        acto_prazo["secretaria"].str.contains("SEINFRA")
        & ~acto_prazo["status"].isin(["Em atendimento", "Pendente"])
        & acto_prazo["responsavel_execucao"].isna(),
        "Execução própria",
        acto_prazo["responsavel_execucao"]
    )

    # Seleção das colunas de interesse
    acto_prazo = acto_prazo[
        [
            "n_da_solicitacao",
            "data_de_solicitacao",
            "mes_solicitacao",
            "ano_solicitacao",
            "servico",
            "solicitante",
            "nome_interessado",
            "cpf_interessado",
            "nome_do_servico",
            "status",
            "etapa_atual",
            "bairro_interessado",
            "tipo_logradouro",
            "executor_atual",
            "bairro_ocorrencia",
            "data_de_finalizacao",
            "mes_finalizacao",
            "ano_finalizacao",
            "bairro_cet",
            "vencimento_sla",
            "tipo_de_manifestacao",
            "tipo_de_registro",
            "canal",
            "bairro_consolidado",
            "prazo_de_conclusao",
            "secretaria",
            "regiao",
            "data_de_vencimento_prazo",
            "mes_vencimento_prazo",
            "ano_vencimento_prazo",
            "tempo_de_execucao_real",
            "dias_ate_vencimento",
            "status_conclusao_servico",
            "unidade_executora",
            "ordem_etapa",
            "protocolo",
            "resposta_secretaria",
            "nome_do_servico_avaliado",
            "e-mail_do_interessado",
            "classificacao_servico_prestado",
            "obs_classificacao_de_servico_prestado",
            "classificacao_atendimento",
            "obs_classificacao_atendimento",
            "questao_resolvida",
            "expectativas",
            "responsavel_execucao"
        ]
    ]

    return acto_prazo


def harmonizar_nome_bairros(df):
    df['bairro_consolidado'] = (
        df['bairro_consolidado']
            .str.replace("Radio", "Rádio")
            .str.replace("Ponta Da Praia", "Ponta da Praia")
            .str.replace("Porta Da Praia", "Ponta da Praia")
            .str.replace("Mr.", "Morro")
            .str.replace("Mor.", "Morro")
            .str.replace("Morro Monte Serrat", "Monte Serrat")
            .str.replace("Boqueirao", "Boqueirão")
            .str.replace("Pompeia", "Pompéia")
            .str.replace("Porto Da Praia", "Porto Ponta Da Praia")
            .str.replace("Morro Jose Menino", "Morro José Menino")
            .str.replace("Morro Da Caneleira", "Morro Caneleira")
            .str.replace("Chico De Paula", "Chico de Paula")
            .str.replace("Ilheu Alto", "Ilhéu Alto")
            .str.replace("Vila Matias", "Vila Mathias")
            .str.replace("Iriri Alto", "Iriri")
            .str.replace("Ilha Barnabé", "Barnabe")
    )
    return df


def main():
    
    acto = processar_os()
    prazo = processar_prazo()
    bairros = processar_bairros()

    acto_prazo = aplicar_merge(acto, prazo, bairros)

    acto_prazo = remover_registros_teste(acto_prazo)
    acto_prazo = tratar_datas(acto_prazo)
    acto_prazo = harmonizar_ordenar_etapas(acto_prazo)
    acto_prazo = tratar_base_final_solicitacoes(acto_prazo)

    acto_prazo = harmonizar_nome_bairros(acto_prazo)

    return acto_prazo
    # return acto


acto_prazo = main()
# acto = main()

StatementMeta(, 4b365623-e19c-4418-9886-8f46267e169f, 7, Finished, Available, Finished)

In [ ]:
acto_prazo.to_csv("/lakehouse/default/Files/acto/acto_prazo.csv")

spark_df = spark.createDataFrame(acto_prazo)
spark_df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .save("Tables/tb_os_acto")

spark.sql("""
CREATE OR REPLACE TABLE tb_os_acto
USING DELTA
AS SELECT * FROM delta.`Tables/tb_os_acto`
""")

StatementMeta(, 4b365623-e19c-4418-9886-8f46267e169f, 8, Finished, Available, Finished)

DataFrame[]